<a href="https://colab.research.google.com/github/Eng7ouda06/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Eng7ouda06/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My rule from Week 4 answers "which pages have a CTR gap right now" -- a snapshot read of
the same 90-day window it ranks. It cannot tell me whether a page is a wash tomorrow, and the
notebook's own limits section already flagged this: the rule and the model need to be judged
on a label from elsewhere in time, not the window the rule was built on.

This data ships exactly that label: `is_declining_label` (from `trend_direction == "down"`),
comparing the last 30 days to the 30 before -- a real forward comparison, not the 90-day
snapshot the baseline reads. That makes the task a yes/no classification with an observed
label, so per the toolkit table I start with **Logistic Regression** (readable, gives me
coefficients I can sanity-check) and compare it to a **Random Forest** (stronger, still gives
feature importances). `trend_pct` and `trend_direction` are the label's own ingredients and
never enter the feature set, along with `impressions_last_30d` / `clicks_last_30d` /
`sessions_last_30d` and their `prev_30d` twins -- those are exactly the columns the label is
built from, so using them as features would just be handing the model the answer.

54.2% of all pages are "declining" by this definition -- a near-coin-flip base rate, which is
the number every model and the baseline both have to beat.

In [ ]:
import os, json, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

DATA = "data/raw/content_refresh_anonymized.csv"
if not Path(DATA).exists():                       # in Colab: fetch the repo, the data ships inside it
    if not Path("flyrank-ml-internship").exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Eng7ouda06/flyrank-ml-internship.git"], check=True)
    os.chdir("flyrank-ml-internship")

df = pd.read_csv(DATA)

# Forward-looking label, already shipped in the data: is this page currently
# in decline (last 30 days vs the 30 before)? Never a feature -- only ever the target.
df["is_declining_label"] = (df.trend_direction == "down").astype(int)
print(f"{len(df):,} pages | {df.is_declining_label.mean():.1%} declining (base rate)")
print(df.trend_direction.value_counts())

30,000 pages | 54.2% declining (base rate)
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Grouped by client, same discipline as the data contract and the leakage skill: `client_id`
is a pseudonym for joins and splits, never a feature, and pages from the same client share a
CMS, a niche and a traffic pattern, so a random row split would leak client identity through
the back door. I hold out 20% of the 32 clients entirely (`GroupShuffleSplit`, seed 42) so the
model is judged on clients it has never seen a single page from -- the split I'd want if this
were shipped to score a brand-new client next month.

In [ ]:
numeric_features = ["impressions_90d", "clicks_90d", "sessions_90d", "pageviews_90d", "users_90d",
                    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
                    "days_with_impressions", "days_with_sessions", "content_age_days",
                    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
                    "scroll_rate", "ai_traffic_pct", "search_volume", "competition", "cpc",
                    "word_count", "char_count"]
categorical_features = ["content_type", "main_intent", "competition_level"]

X_num = df[numeric_features].apply(pd.to_numeric, errors="coerce")
cols_with_na = X_num.columns[X_num.isna().any()]           # missingness follows content_type --
missing_flags = X_num[cols_with_na].isna().astype(int).add_suffix("_missing")   # flag it, don't hide it in a 0
X_num = X_num.fillna(0)
X_cat = pd.get_dummies(df[categorical_features].fillna("unknown").astype(str), dummy_na=False)
X = pd.concat([X_num.reset_index(drop=True), missing_flags.reset_index(drop=True), X_cat.reset_index(drop=True)], axis=1)
y = df["is_declining_label"].astype(int)
groups = df["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
Xtr, Xte, ytr, yte = X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx]

print(f"{X.shape[1]} features | train {len(train_idx):,} rows / {groups.iloc[train_idx].nunique()} clients"
      f" | test {len(test_idx):,} rows / {groups.iloc[test_idx].nunique()} clients")
print(f"test base rate: {yte.mean():.1%} (close to the overall {y.mean():.1%}, so the held-out clients aren't a weird slice)")

40 features | train 23,837 rows / 25 clients | test 6,163 rows / 7 clients
test base rate: 51.1% (close to the overall 54.2%, so the held-out clients aren't a weird slice)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

One table, one split, one set of rows for all three scores -- the baseline, Logistic
Regression, and the Random Forest all get evaluated on the exact same held-out clients. I
report precision@20/50/100 (the size of a real review queue) alongside ROC AUC and average
precision, because the skill is explicit that a model can win at one K and lose at another --
that's a finding, not noise to average away.

Baseline score is the identical Week-4 formula (band CTR gap, `FLOOR=1000`, `MAX_POS=20`,
`RATIO=0.5`), recomputed here so it runs on the *same* rows as the model instead of reading
its own `.csv` (which is git-ignored and only reflects whichever window it was last run on).

In [ ]:
FLOOR, MAX_POS, RATIO = 1000, 20, 0.5
BANDS = ["top_3", "page_1", "striking", "page_3_5", "deep"]

def band_of(pos):
    return pd.cut(pos.where(pos > 0), [0, 3, 10, 20, 50, np.inf], right=False, labels=BANDS)

def baseline_score(f):
    """My Week-4 rule, unchanged: reads only impressions, clicks and position."""
    band = band_of(f.avg_position).astype(object)
    ctr = f.clicks_90d / f.impressions_90d * 100
    ok = (f.avg_position > 0) & (f.avg_position < MAX_POS) & (f.impressions_90d >= FLOOR)
    tot = f[ok].groupby(band[ok])[["clicks_90d", "impressions_90d"]].sum()
    typical = (tot.clicks_90d / tot.impressions_90d * 100).to_dict()
    expected = band.map(typical)
    fires = ok & (ctr < RATIO * expected)
    return np.where(fires, f.impressions_90d * (expected - ctr) / 100, 0.0)

df["baseline_score"] = baseline_score(df)

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

def evaluate(y_true, scores):
    return {"precision@20": precision_at_k(y_true, scores, 20),
            "precision@50": precision_at_k(y_true, scores, 50),
            "precision@100": precision_at_k(y_true, scores, 100),
            "roc_auc": roc_auc_score(y_true, scores),
            "avg_precision": average_precision_score(y_true, scores)}

logreg = Pipeline([("scaler", StandardScaler()),
                    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))])
logreg.fit(Xtr, ytr)
p_log = logreg.predict_proba(Xte)[:, 1]

rf = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20,
                             class_weight="balanced_subsample", random_state=42, n_jobs=-1)
rf.fit(Xtr, ytr)
p_rf = rf.predict_proba(Xte)[:, 1]

results = pd.DataFrame({
    "baseline_rule": evaluate(yte, df["baseline_score"].iloc[test_idx].to_numpy()),
    "logistic_regression": evaluate(yte, p_log),
    "random_forest": evaluate(yte, p_rf),
}).T

os.makedirs("work/outputs", exist_ok=True)
results.round(3).to_csv("work/outputs/w05_model_comparison.csv")
print(results.round(3))

                     precision@20  precision@50  precision@100  roc_auc  \
baseline_rule                0.50          0.56           0.54    0.520   
logistic_regression          0.75          0.80           0.74    0.586   
random_forest                0.50          0.52           0.46    0.605   

                     avg_precision  
baseline_rule                0.518  
logistic_regression          0.589  
random_forest                0.583  


Verdict: Logistic Regression wins the metric that matters for a review queue --
precision@20 = 0.75, precision@50 = 0.80, precision@100 = 0.74, all well above the 0.51 base
rate and clearly ahead of the baseline rule (0.50 / 0.56 / 0.54 -- barely better than guessing).
The Random Forest has the best ROC AUC (0.605 vs Logistic Regression's 0.586) but its
precision@K is at or below the base rate, so it separates classes better on average while being
*worse* at nailing the very top of a ranked queue -- exactly the "wins on one metric, loses on
another" case the skill warns about. Since the deliverable here is a review queue, not a
probability estimate, I'd ship Logistic Regression and keep the Random Forest's ranking only as
a feature-importance cross-check.

Either way, both models clear a much lower bar than I'd like to admit: my Week-4 rule is
essentially useless for predicting *future* decline (0.520 AUC, a coin flip), because it was
never built to -- it flags a CTR gap right now, not a page's trajectory.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

What it leans on: the two models don't agree, which is itself the finding. Logistic
Regression's biggest signals are `users_90d` (-1.23) against `sessions_90d` (+1.11) --
essentially a sessions-per-user contrast -- plus `days_with_impressions` (+0.65). The Random
Forest instead leans hardest on raw `days_with_impressions`, `impressions_90d` and
`avg_position`. Only `days_with_impressions` and `word_count` show up in both top-5 lists.
`users_90d`, `sessions_90d`, `pageviews_90d` and `engaged_sessions_90d` are all near-duplicates
of the same GA4 traffic count, so Logistic Regression is likely splitting one real signal
across several correlated coefficients -- a sign to read its individual coefficients loosely
and trust the ranking (precision@K) more than any single weight.

The shared signal (`days_with_impressions`, `impressions_90d`, `avg_position`) makes some sense
directionally -- pages that have been visible longer and rank better have simply had more
90-day windows in which some 30-day stretch looks "down" by chance -- but I can't rule that out
with this dataset alone, so I read the precision@K gains as directional, not proof of a
genuine early-warning signal.

Where it's most wrong: every one of the top-3 false positives I inspected comes from a single
client (`client_f369cb89fc`) -- pages the model rates as declining that are actually flat or
rising. That's a client-level tell, not a page-level one: the model may be partly learning
"this client's typical impressions/position profile" rather than a universal decline pattern --
exactly the kind of confound a client-grouped split is supposed to surface (and did).

The confident misses in the other direction are small pages (13-83,603 impressions, several
under 60) that are genuinely declining -- consistent with Week 4's own finding that a rate is
barely readable below a volume floor. Below that floor, "no strong signal" and "actually fine"
look the same to any model.

**In practice:** use the ranked queue as "review these first," not "these will decline" -- and
treat any page from a single dominant client near the top of the queue with extra skepticism
until I check whether the client, not the page, is driving the score.

In [ ]:
test_df = df.iloc[test_idx].copy()
test_df["p_log"] = p_log
test_df["y"] = yte.values

top50 = test_df.sort_values("p_log", ascending=False).head(50)
print(f"top-50 by Logistic Regression probability: hit rate {top50.y.mean():.0%}")

coefs = pd.Series(logreg.named_steps["clf"].coef_[0], index=X.columns).sort_values(key=abs, ascending=False)
print("\ntop 5 coefficients (Logistic Regression, standardized features):")
print(coefs.head(5).round(3))

imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\ntop 5 feature importances (Random Forest, cross-check):")
print(imp.head(5).round(3))

cols = ["content_id", "client_id", "p_log", "impressions_90d", "avg_position",
        "content_age_days", "days_since_last_update", "trend_direction"]
print("\nconfident but WRONG -- predicted declining, actually up/stable:")
print(test_df[test_df.y == 0].sort_values("p_log", ascending=False).head(3)[cols].to_string(index=False))

print("\nconfident but WRONG the other way -- predicted safe, actually declining:")
print(test_df[test_df.y == 1].sort_values("p_log", ascending=True).head(3)[cols].to_string(index=False))

top-50 by Logistic Regression probability: hit rate 80%

top 5 coefficients (Logistic Regression, standardized features):
users_90d               -1.234
sessions_90d             1.108
days_with_impressions    0.649
days_with_sessions      -0.470
word_count               0.273
dtype: float64

top 5 feature importances (Random Forest, cross-check):
days_with_impressions    0.157
impressions_90d          0.147
avg_position             0.135
content_age_days         0.108
word_count               0.059
dtype: float64

confident but WRONG -- predicted declining, actually up/stable:
          content_id         client_id    p_log  impressions_90d  avg_position  content_age_days  days_since_last_update trend_direction
content_374e795aab68 client_f369cb89fc 0.885565              235          31.0               181                      20          stable
content_26d48a980581 client_f369cb89fc 0.880396             1266           4.6               106                     106              up
conte